In [1]:
# !pip install ipywidgets
# !pip install jupyterlab_widgets


In [2]:
from collections import deque
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from tqdm.notebook import trange
import copy

In [3]:
import torch
import torch.nn as nn

class Resnet(nn.Module):

    def __init__(self, in_channels, out_channels, resnet_blocks = 5):
        super(Resnet, self).__init__()
        self.start = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size = 3, padding = 1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU()
        )

        self.blocks = nn.ModuleList(
            [ResidualConnection(out_channels) for _ in range(resnet_blocks)]
        )

        self.policy_head = nn.Sequential(
            nn.Conv2d(out_channels, out_channels = 32, kernel_size = 3, padding = 1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(32 * 6 * 7, 7)
        )

        self.value_head = nn.Sequential(
            nn.Conv2d(out_channels, out_channels = 3, kernel_size = 3, padding = 1),
            nn.BatchNorm2d(3),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(3 * 6 * 7, 256),
            nn.ReLU(),
            nn.Linear(256, 1),
            nn.Tanh()
        )

    def forward(self, x):
        x = self.start(x)
        for block in self.blocks:
            x = block(x)
        policy = self.policy_head(x)
        value = self.value_head(x)
        return policy, value
        

class ResidualConnection(nn.Module):

    def __init__(self, out_channels):
        super(ResidualConnection, self).__init__()

        self.block = nn.Sequential(
            nn.Conv2d(in_channels = out_channels, out_channels = out_channels, kernel_size = 3, padding = 1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.Conv2d(in_channels = out_channels, out_channels = out_channels, kernel_size = 3, padding = 1),
            nn.BatchNorm2d(out_channels)     
        )

    def forward(self, x):
        features = self.block(x)
        return torch.relu(features + x)

In [4]:
import numpy as np

class Connect4:

    def __init__(self):
        self.W = 7
        self.H = 6
        self.board = np.zeros((self.H, self.W))
        self.all_moves = np.arange(self.W)


    def reset(self):
        self.board = np.zeros((self.H, self.W))
        return self.board
        
    def get_valid_moves(self, state):
        return (state[0] == 0).astype(int)

    def step(self, board, move, player):
        board = board.copy()
        bcol = move
        brow = self.H-1
        # row, col = move//self.W, move%self.W
        # brow, bcol = self.H-1, col
        while True:
            if board[brow, bcol] == 0:
                board[brow, bcol] = player
                break
            brow -= 1
        done, result = self.is_terminal(board)
        return board, result, done

    def is_terminal(self, board,H = 6, W = 7):
        for r in range(H):
            for c in range(W - 3):
                val = board[r, c]
                if val in [1, -1] and val == board[r, c+1] == board[r, c+2] == board[r, c+3]:
                    return True, val
        for r in range(H - 3):
            for c in range(W):
                val = board[r, c]
                if val in [1, -1] and val == board[r+1, c] == board[r+2, c] == board[r+3, c]:
                    return True, val
        for r in range(H - 3):
            for c in range(W - 3):
                val = board[r, c]
                if val in [1, -1] and val == board[r+1, c+1] == board[r+2, c+2] == board[r+3, c+3]:
                    return True, val
        for r in range(3, H):
            for c in range(W - 3):
                val = board[r, c]
                if val in [1, -1] and val == board[r-1, c+1] == board[r-2, c+2] == board[r-3, c+3]:
                    return True, val
        if not (board == 0).any():
            return True, -0.1
        return False, 0
        
    def stackedStates(self, state, current_player):
        
        return np.stack((
            state == current_player, 
            state == -current_player,
            state == 0
        )).astype(np.float32)

    
    def show(self, state):
        print(state)

In [5]:
import numpy as np

class Node:

    def __init__(self, state, parent, move, player, prob = 0):
        self.H = 6
        self.W = 7
        self.state = state
        self.parent = parent
        self.move = move

        self.player = player
        self.children = {}
        
        self.prob = prob
        
        self.N = 0
        self.W = 0
        
    def is_fully_expanded(self):
        return len(np.argwhere(self.state[0] == 0)) == 0
# returns True if all nodes are expanded else False
    
    def UCB1(self, c = 2):
        if self.N == 0:
            return float('inf')

        # Not sure about this change
        # (Have to look later too)
        q_value = -self.W / self.N
        return q_value + c * self.prob * np.sqrt(self.parent.N)/(self.N + 1)

In [6]:
import torch
# from MCTS_node import Node
import numpy as np

class MCTS:

    def __init__(self, env, root_state, root_player, model, device, PARAMS, which = None):
        self.env = env
        self.H = 6
        self.W = 7
        self.policy = 0
        self.model = model
        self.device = device
        self.root = Node(state = root_state,
                         parent = None,
                         move = None,
                         player = root_player
                        )
        self.PARAMS = PARAMS
        self.which = which
        
    def selection(self):
        current = self.root
        while True:
            is_terminal, _ = self.env.is_terminal(current.state)
            if is_terminal:
                return current
                
            if len(current.children) == 0:
                return current
                
            # if not current.is_fully_expanded():
            #     print(f"Returend From Selection : {current}")
            #     return current
                
            best_child = max(current.children.values(), key = lambda c: c.UCB1())
            current = best_child
            
            
        # return current

    def expansion(self, node, policy):
        valid_states = self.env.get_valid_moves(node.state)

        for action, prob in enumerate(policy):
            if valid_states[action] == 1 and action not in node.children:
                new_state = self.env.step(node.state, action, node.player)[0]
                child = Node(state = new_state,
                             parent = node,
                             move = action,
                             player = -node.player,
                             prob = prob
                            )
                node.children[action] = child
                
    def backpropagation(self, child, value):
        current = child

        while current is not None:
            current.N += 1
            current.W += value
            value = -value
            current = current.parent

    @torch.no_grad()
    def search(self):

        searches = self.PARAMS["SEARCHES"]
        if self.which is not None:
            searches = self.PARAMS["EVALUATION_SEARCHES"]
        
        
        for _ in range(searches):
            node = self.selection()
            is_terminal, value = self.env.is_terminal(node.state)
            
            if is_terminal:
                value = value * node.player
            else:
                stackedStates = self.env.stackedStates(node.state, node.player)
                policy, value = self.model(torch.tensor(stackedStates, dtype = torch.float).unsqueeze(0).to(self.device))
                policy = torch.softmax(policy, dim = 1).squeeze(0).cpu().detach().numpy()
                if node is self.root:
                    alpha = self.PARAMS['ALPHA']
                    eps = self.PARAMS['EPSILON']
                    noise = np.random.dirichlet([alpha] * len(self.env.all_moves))
                    policy = (1 - eps) * policy + eps * noise

                value = value.squeeze(0).cpu().item()
                valid_moves = self.env.get_valid_moves(node.state)
                policy = policy * valid_moves
                if np.sum(policy)>0:
                    policy = policy/(np.sum(policy))
                else:
                    policy = valid_moves/np.sum(valid_moves)


                self.expansion(node, policy)
                # if len(node.children) > 0:
                #     best_action = max(node.children.keys(), key=lambda a: node.children[a].prob)
                #     child = node.children[best_action]
                # else:
                #     child = node
            target_node = node
    
            self.backpropagation(target_node, value)
        actions = np.zeros(7)

        if len(self.root.children) == 0:
            valid_moves = self.env.get_valid_moves(self.root.state)
            action_probs = np.array(valid_moves / np.sum(valid_moves))
            return action_probs

        total_visits = sum(child.N for child in self.root.children.values())
        if total_visits > 0:
            for action, child in self.root.children.items():
                actions[action] = child.N
            actions /= total_visits
        else:
            valid_moves = self.env.get_valid_moves(self.root.state)
            actions = np.array(valid_moves / np.sum(valid_moves))

        return actions

    def __repr__(self):
        return f"{self.root.state}\n{self.root.parent}\n{self.root.children}\n{self.root.action}\n{self.root.children.n}"

In [7]:
from collections import deque
import copy
import numpy as np
# from tqdm import range
import random
import time
# from MCTS import MCTS

import torch.nn as nn
import torch
from torch import optim

class AlphaZero:

    def __init__(self, env, model, policy_loss, value_loss, optimizer, device, PARAMS, current_player = 1):
        self.env = env
        self.model = model
        self.model.to(device)
        self.optimizer = optimizer
        self.mainBuffer = deque(maxlen = 20_000)
        self.policy_loss = policy_loss
        self.value_loss = value_loss
        self.device = device
        self.current_player = current_player
        self.tau = 1.25
        self.entropy_weight = 0.01
        self.PARAMS = PARAMS
        self.entropy_weight = 0.01

        self.avg_loss_per_epoch = []
        self.policy_loss_per_epoch = []
        self.value_loss_per_epoch = []
        self.entropy_per_epoch = []
        self.win_rate = []
        
        self.baseline_model = copy.deepcopy(model)
        self.baseline_model.to(device)
        self.baseline_model.eval()
        for p in self.baseline_model.parameters():
            p.requires_grad = False       
            
        self.scheduler = optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.97)

    def selfPlayData(self, ):
        buffer = []
        state = self.env.reset()
        current_player = 1
        move_number = 0
        # tau = self.PARAMS['TAU']
        while True:
            if move_number < 10:
                tau = 1
            else:
                tau = 0.01

            root = MCTS(self.env, state, current_player, self.model, self.device, self.PARAMS)

            actions = root.search()
            temperature_probs = actions ** (1/tau)
            temperature_probs /= np.sum(temperature_probs)
            buffer.append((state, actions, current_player))

            # ADDED HORIZONTAL FLIP TO DOUBLE THE TRAINING DATA
            flipped_state = np.flip(state, axis=1) 
            flipped_actions = np.flip(actions) 
            buffer.append((flipped_state, flipped_actions, current_player))

            action = np.random.choice(self.env.all_moves, p = temperature_probs)
            state, result, done = self.env.step(state, action, current_player)
            if done:
                mainBuffer = []
                for states, actions, player in buffer:
                    if result == 0:
                        value = 0
                    elif result == player:
                        value = 1
                    else:
                        value = -1
                    mainBuffer.append((self.env.stackedStates(states, player), actions, value))
                return mainBuffer

            current_player = -current_player
            move_number += 1

    def train(self):
        for i in trange(self.PARAMS['TOTAL_ITERATIONS']):
            
            print("==" *90)
            print(f"Training Iteration {i}")
            self.model.eval()

            for _ in trange(self.PARAMS['SELF_PLAY_ITERATIONS']):
                self.mainBuffer.extend(self.selfPlayData())
            
            self.model.train()



            sum_epoch_loss = 0
            sum_policy_loss = 0
            sum_value_loss = 0
            sum_entropy = 0

            for epoch in trange(self.PARAMS['EPOCHS']):

                epoch_loss = 0
                batch_count = 0
                np.random.shuffle(self.mainBuffer)
                for idx in range(0, len(self.mainBuffer), self.PARAMS["BATCH_SIZE"]):
                    # print(self.mainBuffer)
                    # print(idx)
                    data = random.sample(self.mainBuffer, self.PARAMS["BATCH_SIZE"])
                    states, actions, value = zip(*data)
    
                    actions = torch.tensor(np.array(actions), dtype = torch.float).to(self.device)
                    values = torch.tensor(np.array(value), dtype = torch.float).unsqueeze(1).to(self.device)
                    states = torch.tensor(np.array(states), dtype = torch.float).to(self.device)

                    model_policy, model_value = self.model(states)
                    
                    policy_probs = torch.softmax(model_policy, dim=1)
                    entropy = -torch.sum(policy_probs * torch.log(policy_probs + 1e-10), dim=1).mean()
                    
                    policy_loss = self.policy_loss(model_policy, actions)
                    value_loss = self.value_loss(model_value, values)
                    
                    
                    total_loss = policy_loss + value_loss - self.entropy_weight * entropy
                    
                    self.optimizer.zero_grad()
                    total_loss.backward()
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                    self.optimizer.step()            

                    sum_epoch_loss += total_loss.item()
                    sum_policy_loss += policy_loss.item()
                    sum_value_loss += value_loss.item()
                    sum_entropy += entropy.item()

                    batch_count += 1

                avg_loss = sum_epoch_loss/batch_count
                
                    
                print(f"\tEpoch {epoch}\tTotal Loss : {avg_loss}")

                self.avg_loss_per_epoch.append(avg_loss)
                self.policy_loss_per_epoch.append(sum_policy_loss/batch_count)
                self.value_loss_per_epoch.append(sum_value_loss/batch_count)
                self.entropy_per_epoch.append(sum_entropy/batch_count)


                
            print("\nEVALUATION PHASE------------")
            self.evaluation_low()
            self.evaluation()

            # if win_rate > 0.45:
            self.save_model(i)

            # if win_rate > 0.55:
            # print("Baseline Model Changed!!!")
            self.baseline_model = copy.deepcopy(self.model)
            self.baseline_model.to(self.device)
            self.baseline_model.eval()
            for p in self.baseline_model.parameters():
                p.requires_grad = False

    def save_model(self, i):

        checkpoint = {
            "model" : self.model.state_dict(),
            "optimizer_state" : self.optimizer.state_dict()
        }
        torch.save(checkpoint, f'Checkpoints/checkpoint{i}.pth')
        print(f"\nSaved Checkpoint {i}!!\n")
        print("==" * 90)
        print("\n\n\n")
        

    def play_game(self, model1, model2):
        state = self.env.reset()
        current_player = 1

        while True:
            root = MCTS(self.env, state, current_player, model1, self.device, self.PARAMS, 1)
            actions = root.search()
            action = np.argmax(actions)
            state, result, done = self.env.step(state, action, current_player)
            if done:
                return result * current_player
            current_player = -current_player

            root = MCTS(self.env, state, current_player, model2, self.device, self.PARAMS, 1)
            actions = root.search()
            action = np.argmax(actions)
            state, result, done = self.env.step(state, action, current_player)
            if done:
                return result * -current_player
            current_player = -current_player
    

    def evaluation(self):
        wins = 0
        draws = 0
        self.model.eval()
        pmodel = self.baseline_model

        for game in trange(self.PARAMS['EVALUATION_GAMES']):
            if game%2 == 0:
                result = self.play_game(self.model, pmodel)
            else:
                result = -self.play_game(pmodel, self.model)

            if result == 1:
                wins += 1
            elif result == 0:
                draws += 1

        win_rate = wins/self.PARAMS["EVALUATION_GAMES"]
        self.win_rate.append(win_rate)
        # print("=" * 70)
        print(f"\n\nEvaluation : {wins}/{self.PARAMS['EVALUATION_GAMES']} \nRate : {win_rate} \nDraws : {draws}")
        # return win_rate

    def random_agent(self, model1):
        state = self.env.reset()
        current_player = 1

        while True:
            root = MCTS(self.env, state, current_player, model1, self.device, self.PARAMS, 1)
            actions = root.search()
            action = np.argmax(actions)
            state, result, done = self.env.step(state, action, current_player)
            if done:
                return result * current_player
            current_player = -current_player

            # root = MCTS(self.env, state, current_player, model2, self.device, self.PARAMS, 1)
            # actions = root.search()
            # action = np.argmax(actions)
            actions = np.random.choice(self.env.get_valid_moves(state))
            state, result, done = self.env.step(state, action, current_player)
            if done:
                return result * -current_player
            current_player = -current_player

    def evaluation_low(self):
        wins = 0
        draws = 0
        self.model.eval()
        pmodel = self.baseline_model

        for game in trange(self.PARAMS['EVALUATION_GAMES']):
            if game%2 == 0:
                result = self.play_game(self.model, pmodel)
            else:
                result = self.random_agent(self.model)

            if result == 1:
                wins += 1
            elif result == 0:
                draws += 1


        win_rate = wins/self.PARAMS["EVALUATION_GAMES"]
        print("=" * 50)
        print(f"\nEvaluation with random Agent: {wins}/{self.PARAMS['EVALUATION_GAMES']} \nRate : {win_rate} \nDraws : {draws}")
        # return win_rate
                            

In [8]:
# PARAMETERS = {
    
#     "IN_CHANNELS" : 3,
#     "OUT_CHANNELS" : 64,
#     "RESNET_BLOCKS" : 5,
    
#     "SELF_PLAY_ITERATIONS" : 50,
#     "EPOCHS" : 10,
#     "BATCH_SIZE" : 128,
#     "TOTAL_ITERATIONS" : 200,
    
#     "SEARCHES" : 300,
#     "EVALUATION_GAMES" : 50,
#     "EVALUATION_SEARCHES" : 600,
    
#     "ALPHA" : 0.6,
#     "EPSILON" : 0.25,
#     "TAU" : 1
#     }


In [9]:

from torch import optim
import numpy as np
import torch
import torch.nn as nn

PARAMETERS = {
    
    "IN_CHANNELS" : 3,
    "OUT_CHANNELS" : 128,
    "RESNET_BLOCKS" : 5,
    
    "SELF_PLAY_ITERATIONS" : 100, #100
    "EPOCHS" : 5,
    "BATCH_SIZE" : 128,
    "TOTAL_ITERATIONS" : 150,  #200
    
    "SEARCHES" : 800,
    "EVALUATION_GAMES" : 50,
    "EVALUATION_SEARCHES" : 600,
    
    
    "ALPHA" : 1.4,
    "EPSILON" : 0.25,
    "TAU" : 1
    }


env = Connect4()

model = Resnet(PARAMETERS['IN_CHANNELS'], 
               PARAMETERS['OUT_CHANNELS'], 
               PARAMETERS['RESNET_BLOCKS']
              )

optimizer = optim.Adam(model.parameters(), lr = 0.0008, weight_decay = 1e-4)
policy_loss = nn.CrossEntropyLoss()
value_loss = nn.MSELoss()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
 
agent= AlphaZero(env, 
                 model, 
                 policy_loss, 
                 value_loss, 
                 optimizer, 
                 device, 
                 PARAMETERS
                )

agent.train()


  0%|          | 0/150 [00:00<?, ?it/s]

Training Iteration 0


  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

	Epoch 0	Total Loss : 2.655435800552368
	Epoch 1	Total Loss : 4.925823962688446
	Epoch 2	Total Loss : 7.008238807320595
	Epoch 3	Total Loss : 8.97799111008644
	Epoch 4	Total Loss : 10.894720321893692

EVALUATION PHASE------------


...


Evaluation with random Agent: 50/50 
Rate : 1.0 
Draws : 0


  0%|          | 0/50 [00:00<?, ?it/s]



Evaluation : 49/50 
Rate : 0.98 
Draws : 0

Saved Checkpoint 21!!





Training Iteration 22


  0%|          | 0/100 [00:00<?, ?it/s]

AcceleratorError: CUDA error: unspecified launch failure
Search for `cudaErrorLaunchFailure' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
# from Resnet_model import Resnet
# from connect4 import Connect4
# from torch import optim
# import numpy as np
# import torch
# # from alphazero import AlphaZero
# import torch.nn as nn

PARAMETERS = {
    
    "IN_CHANNELS" : 3,
    "OUT_CHANNELS" : 128,
    "RESNET_BLOCKS" : 5,
    
    "SELF_PLAY_ITERATIONS" : 100, #100
    "EPOCHS" : 5,
    "BATCH_SIZE" : 128,
    "TOTAL_ITERATIONS" : 100,  #200
    
    "SEARCHES" : 800,
    "EVALUATION_GAMES" : 50,
    "EVALUATION_SEARCHES" : 600,
    
    
    "ALPHA" : 1.4,
    "EPSILON" : 0.25,
    "TAU" : 1
    }


env = Connect4()

# model = Resnet(PARAMETERS['IN_CHANNELS'], 
#                PARAMETERS['OUT_CHANNELS'], 
#                PARAMETERS['RESNET_BLOCKS']
#               )

# optimizer = optim.Adam(model.parameters(), lr = 0.0008, weight_decay = 1e-4)
# policy_loss = nn.CrossEntropyLoss()
# value_loss = nn.MSELoss()
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
 
agent= AlphaZero(env, 
                 model, 
                 policy_loss, 
                 value_loss, 
                 optimizer, 
                 device, 
                 PARAMETERS
                )

agent.train()
